# Практическая работа. Иерархическая кластеризация, DBSCAN и оценка качества кластеризации
## «Не шары, не k и не без шума»

**Цель:** научиться оценивать кластеризацию без ответов (коэффициент силуэта) и с ответами (скорректированный индекс Рэнда), написать на чистом Python агломеративную кластеризацию с тремя способами связи и метод DBSCAN, разделить «полумесяцы», на которых метод k средних ошибается, и найти ложные отражения в облаке точек лидара.

> **Правило работы.** Внутри своих функций – только списки, словари, циклы, `math` и `random`. Библиотеки `sklearn` и `scipy` нужны только для загрузки Wine и сверки, `matplotlib` – только в готовых ячейках с графиками.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом – подсказка написана в комментарии `# <--`. Напечатанные значения должны совпасть с тем, что стоит после слова «ожидается». Готовые ячейки просто выполните. Сначала прочитайте теорию.

**План работы**

| Задание | Что пишем | Главный результат |
|---|---|---|
| Г1 | `silhouette_samples`, `silhouette_score` | силуэт выбирает 6 объектов на парковке и 3 сорта вина |
| Г2 | `adjusted_rand_index` | чистота растет с числом кластеров, а индекс Рэнда – нет |
| Г3 | `agglomerative` | дерево слияний на пяти точках |
| Г4 | способы связи на «полумесяцах» | одиночная связь разделяет их без ошибок |
| Г5 | `region_query`, `dbscan`, `point_types` | основные, пограничные и шумовые точки |
| Г6 | DBSCAN против k средних | «полумесяцы»: 1,000 против 0,246 |
| Г7 | график k-расстояний, сцена с шумом | ложные отражения лидара найдены без числа кластеров |

## Теория: другие методы кластеризации и оценка качества

**1. Зачем другие методы.** На прошлом занятии мы видели ограничения метода k средних: число кластеров нужно задать заранее, кластеры получаются «шарами» с прямыми границами, а каждый объект, даже случайный выброс, обязательно попадает в какой-то кластер. Сегодня – два метода без этих ограничений и две меры качества.

**2. Агломеративная (иерархическая) кластеризация.** Идея «снизу вверх»:

1. сначала каждый объект – отдельный кластер;
2. найти два **ближайших** кластера и объединить их;
3. повторять, пока не останется один кластер.

Записав, какие кластеры и на каком расстоянии объединялись, получаем **дендрограмму** – дерево слияний, где высота соединения равна расстоянию между кластерами в момент слияния. Чтобы получить $k$ кластеров, дерево «разрезают» горизонтальной линией: либо останавливаются, когда кластеров осталось $k$, либо выбирают высоту разреза. Число кластеров можно выбрать уже после построения дерева, глядя на большие «скачки» высоты.

**3. Способы связи** – как считать расстояние между кластерами $A$ и $B$:

| Способ связи | Расстояние между кластерами | Особенность |
|---|---|---|
| одиночная (single) | расстояние между **ближайшими** точками $A$ и $B$ | находит вытянутые и изогнутые кластеры, но склеивает кластеры через «мостики» из шумовых точек |
| полная (complete) | расстояние между **самыми далекими** точками | компактные кластеры похожего размера |
| средняя (average) | среднее расстояние по всем парам точек из $A$ и $B$ | компромисс между двумя предыдущими |
| Уорда (ward) | на сколько вырастет инерция при объединении | похожа на k средних; способ по умолчанию в sklearn |

После объединения $A$ и $B$ в новый кластер расстояние от него до любого другого кластера $C$ не нужно считать заново по всем точкам: для одиночной связи это $\min(d_{AC}, d_{BC})$, для полной $\max(d_{AC}, d_{BC})$, для средней $\dfrac{n_A d_{AC} + n_B d_{BC}}{n_A + n_B}$, где $n_A$, $n_B$ – размеры кластеров.

Цена: на каждом из $n - 1$ шагов ищется ближайшая пара среди всех кластеров, поэтому простая реализация работает за время порядка $n^3$. Для 100 точек это быстро, для 100 000 – уже нет.

**4. DBSCAN** (кластеризация по плотности). Два гиперпараметра: радиус $\varepsilon$ и минимальное число точек `min_samples`. **$\varepsilon$-окрестность** точки – все точки на расстоянии не больше $\varepsilon$, включая саму точку.

- **Основная** точка – в ее окрестности не меньше `min_samples` точек (точка внутри плотной области).
- **Пограничная** точка – сама не основная, но лежит в окрестности какой-то основной.
- **Шумовая** точка – ни то ни другое; ей ставится метка $-1$.

Кластер – это все точки, до которых можно дойти от основной точки цепочкой шагов длиной не больше $\varepsilon$, причем продолжать цепочку можно только через основные точки. Алгоритм: берем еще не просмотренную точку; если она не основная – пока считаем ее шумом; если основная – начинаем новый кластер и «обходим» его: добавляем всех соседей, а соседей основных соседей добавляем в очередь. Пограничная точка, раньше помеченная шумом, при обходе переходит в кластер.

Плюсы: число кластеров не нужно задавать, форма кластеров любая, выбросы отмечаются как шум. Минусы: один радиус $\varepsilon$ для всех кластеров (кластеры разной плотности плохо разделяются), чувствительность к масштабу признаков, нужно подбирать $\varepsilon$.

**5. Как выбрать $\varepsilon$: график k-расстояний.** Для каждой точки найдем расстояние до ее `min_samples`-го ближайшего соседа (сама точка считается первым соседом с расстоянием 0, как в sklearn) и отсортируем эти расстояния. У точек внутри кластеров они маленькие, у шумовых – большие. На графике виден «локоть»: значение $\varepsilon$ берут около него.

**6. Коэффициент силуэта** – мера качества **без ответов** (внутренняя мера). Для точки $i$:

- $a(i)$ – среднее расстояние до остальных точек **своего** кластера;
- $b(i)$ – среднее расстояние до точек **ближайшего чужого** кластера (среднее считаем для каждого чужого кластера и берем наименьшее);

$$s(i) = \frac{b(i) - a(i)}{\max\big(a(i),\, b(i)\big)}, \qquad -1 \le s(i) \le 1.$$

$s(i)$ близко к 1 – точка далеко от чужих и близко к своим; около 0 – на границе; меньше 0 – скорее всего, не в своем кластере. Коэффициент силуэта кластеризации – среднее $s(i)$ по всем точкам (точки шума DBSCAN не учитываются). Для точки, одной в своем кластере, $s(i) = 0$. Силуэт можно сравнивать при разном числе кластеров. Но он, как и инерция, «любит» компактные выпуклые кластеры.

**7. Скорректированный индекс Рэнда (ARI)** – мера сходства с известными ответами (внешняя мера). Рассмотрим все $\binom{n}{2} = \frac{n(n-1)}{2}$ пар объектов. Пара «согласована», если оба разбиения либо кладут ее вместе, либо разделяют. Доля согласованных пар – индекс Рэнда. Даже случайное разбиение дает заметную долю согласия, поэтому индекс **корректируют на случайность**. Через таблицу сопряженности $n_{ij}$ (кластер $i$, ответ $j$) с суммами по строкам $a_i$ и по столбцам $b_j$:

$$\mathrm{ARI} = \frac{\sum_{ij}\binom{n_{ij}}{2} - E}{\frac{1}{2}\left[\sum_i\binom{a_i}{2} + \sum_j\binom{b_j}{2}\right] - E}, \qquad E = \frac{\sum_i\binom{a_i}{2}\,\sum_j\binom{b_j}{2}}{\binom{n}{2}}.$$

ARI = 1 – разбиения совпадают (номера кластеров не важны), ARI около 0 – совпадение не лучше случайного. В отличие от чистоты (Б2.3), ARI не растет сам по себе от увеличения числа кластеров.

**Где почитать.** Открытый курс машинного обучения ODS (mlcourse.ai), тема 7 «Обучение без учителя: PCA и кластеризация» (статья на Хабре); документация scikit-learn, раздел «Clustering»: Hierarchical clustering, DBSCAN, Silhouette Coefficient, Rand index.

In [ ]:
# Готовая ячейка: библиотеки, данные и функции из Б2.3
import math
import random
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine     # только для загрузки набора Wine

RANDOM_STATE = 42
BLUE, NAVY, GREY, DARK_BLUE = '#4C9BFF', '#0F1B2D', '#969CA3', '#1F63C9'
COLORS = ['#4C9BFF', '#0F1B2D', '#969CA3', '#1F63C9', '#9CC8FF', '#5B6472', '#C9DEFF', '#283343']

def make_scene(seed=RANDOM_STATE, n_noise=0):
    """Парковка из Б2.3: точки лидара от шести объектов. n_noise – ложные отражения,
    равномерно разбросанные по площадке; их номер объекта y = -1."""
    rnd = random.Random(seed)
    objects = [((3, 3), (0.8, 0.5), 50), ((9, 2.5), (0.6, 0.6), 35), ((5, 9), (1.1, 0.6), 60),
               ((13, 9), (0.5, 0.9), 40), ((16, 3), (0.7, 0.7), 45), ((10, 13), (0.9, 0.5), 50)]
    X, y = [], []
    for c, ((cx, cy), (sx, sy), n) in enumerate(objects):
        for _ in range(n):
            X.append([round(rnd.gauss(cx, sx), 2), round(rnd.gauss(cy, sy), 2)])
            y.append(c)
    idx = list(range(len(X)))
    rnd.shuffle(idx)
    X, y = [X[i] for i in idx], [y[i] for i in idx]
    rnd_noise = random.Random(seed + 1)
    for _ in range(n_noise):
        X.append([round(rnd_noise.uniform(0, 19), 2), round(rnd_noise.uniform(0, 15), 2)])
        y.append(-1)
    return X, y

def make_moons(n=200, noise=0.08, seed=RANDOM_STATE):
    """Два вложенных «полумесяца» (Б2.3)."""
    rnd = random.Random(seed)
    X, y = [], []
    for i in range(n):
        t = math.pi * rnd.random()
        if i % 2 == 0:
            X.append([math.cos(t) + rnd.gauss(0, noise), math.sin(t) + rnd.gauss(0, noise)]); y.append(0)
        else:
            X.append([1 - math.cos(t) + rnd.gauss(0, noise), 0.5 - math.sin(t) + rnd.gauss(0, noise)]); y.append(1)
    return X, y

# ---- метод k средних из Б2.3
def dist2(a, b):
    return sum((ai - bi) ** 2 for ai, bi in zip(a, b))

def dist(a, b):
    return math.sqrt(dist2(a, b))

def assign_clusters(X, centers):
    return [min(range(len(centers)), key=lambda j: dist2(x, centers[j])) for x in X]

def update_centers(X, labels, k, old):
    centers = []
    for j in range(k):
        pts = [x for x, l in zip(X, labels) if l == j]
        centers.append([sum(p[d] for p in pts) / len(pts) for d in range(len(X[0]))] if pts else list(old[j]))
    return centers

def kmeans(X, k, init, max_iter=100):
    centers, labels, history = [list(c) for c in init], None, []
    for _ in range(max_iter):
        new = assign_clusters(X, centers)
        if new == labels:
            break
        labels = new
        centers = update_centers(X, labels, k, centers)
        history.append(sum(dist2(x, centers[l]) for x, l in zip(X, labels)))
    return centers, labels, history

def kmeans_pp_init(X, k, rnd):
    centers = [list(rnd.choice(X))]
    while len(centers) < k:
        D2 = [min(dist2(x, c) for c in centers) for x in X]
        r, acc = rnd.random() * sum(D2), 0.0
        for x, d in zip(X, D2):
            acc += d
            if acc >= r:
                centers.append(list(x)); break
    return centers

def kmeans_restarts(X, k, n_init=10, seed=RANDOM_STATE):
    rnd, best = random.Random(seed), None
    for _ in range(n_init):
        c, l, h = kmeans(X, k, kmeans_pp_init(X, k, rnd))
        if best is None or h[-1] < best[2]:
            best = (c, l, h[-1])
    return best

def contingency(labels, y):
    """Таблица сопряженности: строки – кластеры, столбцы – ответы (в порядке возрастания номеров)."""
    rows, cols = sorted(set(labels)), sorted(set(y))
    return [[sum(1 for a, b in zip(labels, y) if a == i and b == j) for j in cols] for i in rows]

def purity(table):
    return sum(max(r) for r in table) / sum(sum(r) for r in table)

def standardize(X):
    n = len(X)
    result = [list(r) for r in X]
    for j in range(len(X[0])):
        mu = sum(r[j] for r in X) / n
        sigma = math.sqrt(sum((r[j] - mu) ** 2 for r in X) / n)
        for r in result:
            r[j] = (r[j] - mu) / sigma
    return result

def plot_clusters(ax, X, labels, title=''):
    """Точки цветом кластера; шум (метка -1) – серыми крестиками."""
    for j in sorted(set(labels)):
        pts = [x for x, l in zip(X, labels) if l == j]
        if j == -1:
            ax.scatter([p[0] for p in pts], [p[1] for p in pts], s=22, marker='x', color=GREY)
        else:
            ax.scatter([p[0] for p in pts], [p[1] for p in pts], s=12, color=COLORS[j % len(COLORS)])
    ax.set_title(title); ax.set_aspect('equal'); ax.grid(alpha=0.3)

X_scene, y_scene = make_scene()
X_moons, y_moons = make_moons()
wine = load_wine()
Z_wine, y_wine = standardize(wine.data.tolist()), wine.target.tolist()
print('парковка:', len(X_scene), ' полумесяцы:', len(X_moons), ' Wine:', len(Z_wine), '  ожидается 280 200 178')

**Задание Г1. Коэффициент силуэта.**

- `silhouette_samples(X, labels)` – список значений $s(i)$ по формуле из п. 6 теории. Для шумовой точки (метка $-1$) запишите `None`; для точки, одной в своем кластере, – `0.0`. Удобно сначала посчитать матрицу расстояний `D[i][j]` один раз.
- `silhouette_score(X, labels)` – среднее $s(i)$ без значений `None`.

Затем выберем число кластеров: для $k$ от 2 до 10 возьмем лучший из 10 запусков k средних (готовая `kmeans_restarts`) и посчитаем силуэт. Сравним с методом «локтя» из Б2.3.

*Считаем на бумаге.* Точки $0, 1$ (кластер 0) и $5$ (кластер 1) на прямой. Для точки 0: $a = 1$, $b = 5$, $s = (5 - 1)/5 = 0{,}8$. Для точки 5 она одна в кластере: $s = 0$.

In [ ]:
def silhouette_samples(X, labels):
    n = len(X)
    D = [[dist(X[i], X[j]) for j in range(n)] for i in range(n)]
    clusters = sorted(set(l for l in labels if l != -1))
    s = []
    for i in range(n):
        if labels[i] == -1:
            s.append(None)                                                   # шум не оцениваем
            continue
        same = [D[i][j] for j in range(n) if labels[j] == labels[i] and j != i]
        if not same:
            s.append(0.0)                                                    # точка одна в кластере
            continue
        a = ...                                 # <-- среднее до своих
        means = ...                             # <-- среднее расстояние до каждого чужого кластера
        b = ...                                 # <-- среднее до ближайшего чужого кластера
        ...                                     # <-- силуэт точки
    return s

def silhouette_score(X, labels):
    values = [v for v in silhouette_samples(X, labels) if v is not None]
    return ...                                  # <-- среднее без шума


print([round(v, 3) for v in silhouette_samples([[0], [1], [5]], [0, 0, 1])], '  ожидается [0.8, 0.75, 0.0]')

print(' k   силуэт')
sil_scene = {}
for k in range(2, 11):
    c, l, J = kmeans_restarts(X_scene, k)
    sil_scene[k] = ...                          # <-- силуэт лучшего разбиения
    print(f'{k:2d}   {sil_scene[k]:.3f}')
print('лучшее k:', max(sil_scene, key=sil_scene.get), '  ожидается 6 (силуэт 0.767)')

sil_wine = {}
for k in range(2, 7):
    c, l, J = kmeans_restarts(Z_wine, k)
    sil_wine[k] = silhouette_score(Z_wine, l)
print('Wine:', {k: round(v, 3) for k, v in sil_wine.items()}, '  ожидается максимум 0.285 при k = 3')

**Вопрос Г1.** а) Почему для точки 1 в примере силуэт 0,75, а не 0,8, как у точки 0? б) В Б2.3 «локоть» на Wine был почти не виден. Что показывает силуэт? в) Силуэт на Wine (0,285) намного меньше, чем на парковке (0,767). Значит ли это, что на Wine кластеризация плохая?

*Ваш ответ:*

**Задание Г2. Скорректированный индекс Рэнда.**

- `comb2(m)` – число пар из $m$ объектов: $\binom{m}{2} = \frac{m(m-1)}{2}$.
- `adjusted_rand_index(labels, y)` – ARI по формуле из п. 7 теории. Таблицу сопряженности дает готовая `contingency(labels, y)`; суммы по строкам – размеры кластеров $a_i$, по столбцам – размеры классов $b_j$.

Затем сравним чистоту (Б2.3) и ARI на Wine при $k$ от 2 до 10.

In [ ]:
def comb2(m):
    return ...                                  # <-- число пар

def adjusted_rand_index(labels, y):
    table = contingency(labels, y)
    pairs_together = ...                        # <-- сумма C(n_ij, 2)
    sum_a = ...                                 # <-- по размерам кластеров
    sum_b = ...                                 # <-- по размерам классов
    expected = ...                              # <-- ожидаемое при случайном разбиении
    max_index = (sum_a + sum_b) / 2
    return (pairs_together - expected) / (max_index - expected)


print(adjusted_rand_index([0, 0, 1, 1], [0, 0, 1, 1]), '  ожидается 1.0')
print(adjusted_rand_index([1, 1, 0, 0], [0, 0, 1, 1]), '  ожидается 1.0: номера кластеров не важны')
print(round(adjusted_rand_index([0, 0, 1, 2], [0, 0, 1, 1]), 3), '  ожидается 0.571')
rnd = random.Random(0)
print(round(adjusted_rand_index([rnd.randrange(3) for _ in y_wine], y_wine), 3), '  ожидается около 0: случайное разбиение')

print(' k   чистота   ARI')
for k in range(2, 11):
    c, l, J = kmeans_restarts(Z_wine, k)
    print(f'{k:2d}    {purity(contingency(l, y_wine)):.3f}    {adjusted_rand_index(l, y_wine):.3f}')
print('  ожидается: k = 3 – 0.966 и 0.897;  k = 4 – 0.972 и 0.761;  k = 10 – 0.949 и 0.375')

In [ ]:
# Готовая ячейка: сверка с sklearn
from sklearn.metrics import adjusted_rand_score, silhouette_score as sk_silhouette
c, l, J = kmeans_restarts(Z_wine, 3)
print(f'ARI:    наш {adjusted_rand_index(l, y_wine):.6f},  sklearn {adjusted_rand_score(y_wine, l):.6f}')
print(f'силуэт: наш {silhouette_score(Z_wine, l):.6f},  sklearn {sk_silhouette(Z_wine, l):.6f}')

**Вопрос Г2.** а) При $k = 4$ чистота (0,972) даже выше, чем при $k = 3$, и при любом $k$ до 10 почти не падает, хотя десять кластеров для трех сортов – явно хуже. Почему? Чему равна чистота при $k = 178$? б) Как ARI «наказывает» лишние кластеры? в) Можно ли посчитать ARI для парковки из Б2.3? А для настоящих данных лидара с дороги?

*Ваш ответ:*

**Задание Г3. Агломеративная кластеризация.**

`agglomerative(X, n_clusters, linkage)` по п. 2–3 теории:

1. кластеры храним в словаре `members`: номер кластера → список номеров точек; вначале кластер $i$ – это точка $i$;
2. расстояния между кластерами храним в словаре `D` с ключом `(меньший номер, больший номер)`;
3. на каждом шаге среди активных кластеров найти пару с наименьшим расстоянием, объединить ее в **новый** кластер с номером `next_id` (номера $n, n+1, \dots$ – как в scipy), записать в `history` четверку `(a, b, расстояние, размер нового кластера)`;
4. расстояния от нового кластера до остальных пересчитать по правилу способа связи (`'single'`, `'complete'`, `'average'`);
5. в момент, когда активных кластеров стало `n_clusters`, запомнить их состав.

Функция возвращает `labels` (номер кластера для каждой точки при `n_clusters` кластерах) и полную `history` – до одного кластера, чтобы нарисовать дендрограмму.

*Считаем на бумаге.* Точки на прямой: 0, 1, 3, 7, 8. Сначала объединяются $\{0, 1\}$ и $\{7, 8\}$ (расстояние 1). Затем $\{0, 1\}$ и $\{3\}$: одиночная связь дает $\min(3, 2) = 2$, полная $\max(3, 2) = 3$, средняя $(3 + 2)/2 = 2{,}5$. Последнее слияние: одиночная – $7 - 3 = 4$, полная – $8 - 0 = 8$, средняя – среднее шести расстояний $= 37/6 \approx 6{,}17$.

In [ ]:
def agglomerative(X, n_clusters, linkage='single'):
    n = len(X)
    members = {i: [i] for i in range(n)}
    D = {}
    for i in range(n):
        for j in range(i + 1, n):
            D[(i, j)] = dist(X[i], X[j])
    active, next_id, history = list(range(n)), n, []
    snapshot = [[i] for i in range(n)] if n_clusters >= n else None
    while len(active) > 1:
        best = None
        for p in range(len(active)):
            for q in range(p + 1, len(active)):
                a, b = active[p], active[q]
                d = D[(min(a, b), max(a, b))]
                if best is None or d < best[2]:
                    best = ...                  # <-- ближайшая пара кластеров
        a, b, d = best
        new = next_id
        next_id += 1
        members[new] = ...                      # <-- состав нового кластера
        history.append((a, b, d, len(members[new])))
        active = [c for c in active if c not in (a, b)]
        for c in active:
            da, db = D[(min(a, c), max(a, c))], D[(min(b, c), max(b, c))]
            if linkage == 'single':
                ...                             # <-- одиночная связь
            elif linkage == 'complete':
                ...                             # <-- полная связь
            else:
                na, nb = len(members[a]), len(members[b])
                ...                             # <-- средняя связь
        active.append(new)
        if len(active) == n_clusters:
            snapshot = [list(members[c]) for c in active]
    labels = [0] * n
    for k, group in enumerate(snapshot):
        for i in group:
            labels[i] = k
    return labels, history


T = [[0], [1], [3], [7], [8]]
for link in ('single', 'complete', 'average'):
    labels, history = agglomerative(T, 2, link)
    print(f'{link:8s}  высоты слияний: {[round(h[2], 2) for h in history]}   кластеры: {labels}')
print('  ожидается: single [1.0, 1.0, 2.0, 4.0];  complete [1.0, 1.0, 3.0, 8.0];  average [1.0, 1.0, 2.5, 6.17]')

**Вопрос Г3.** а) Почему при одиночной и полной связи кластеры на пяти точках получились одинаковыми, а высоты слияний разные? б) Почему при объединении кластеров размер `members[new]` важен для средней связи, но не нужен для одиночной и полной? в) Как по высотам слияний понять, сколько кластеров «естественно» в данных?

*Ваш ответ:*

**Задание Г4. Способы связи на «полумесяцах».**

Возьмем 100 точек «полумесяцев» (агломеративная кластеризация медленная, 200 точек считались бы дольше) и сравним три способа связи по ARI. Затем сверим высоты слияний с функцией `linkage` из scipy и нарисуем дендрограммы.

In [ ]:
X_m100, y_m100 = make_moons(n=100)
agg = {}
for link in ('single', 'complete', 'average'):
    labels, history = ...                       # <-- два кластера выбранным способом связи
    agg[link] = (labels, history)
    print(f'{link:8s}  ARI = {adjusted_rand_index(labels, y_m100):.3f},  размеры кластеров {sorted([labels.count(0), labels.count(1)])}')
print('  ожидается: single 1.000 [50, 50];  complete 0.379;  average 0.457')

In [ ]:
# Готовая ячейка: сверка с scipy и дендрограммы
from scipy.cluster.hierarchy import linkage, dendrogram
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, link in zip(ax, ('single', 'complete', 'average')):
    Z = linkage(X_m100, method=link)
    diff = max(abs(h[2] - z[2]) for h, z in zip(agg[link][1], Z))
    print(f'{link:8s}: наибольшее расхождение высот со scipy {diff:.2e}   ожидается около 1e-16')
    ours = [[h[0], h[1], h[2], h[3]] for h in agg[link][1]]
    dendrogram(ours, ax=a, no_labels=True, color_threshold=0, above_threshold_color=NAVY)
    a.set_title(f'{link}'); a.set_ylabel('высота слияния')
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
for a, link in zip(ax, ('single', 'complete', 'average')):
    plot_clusters(a, X_m100, agg[link][0], link)
plt.tight_layout(); plt.show()

**Вопрос Г4.** а) Почему одиночная связь разделила «полумесяцы» без ошибок? б) Посмотрите на дендрограмму одиночной связи: чем последнее слияние отличается от остальных? в) Представьте, что между полумесяцами разбросано несколько случайных точек. Что случится с одиночной связью?

*Ваш ответ:*

**Задание Г5. DBSCAN.**

- `region_query(X, i, eps)` – номера всех точек на расстоянии не больше `eps` от точки `i`, включая саму `i`.
- `dbscan(X, eps, min_samples)` по п. 4 теории:
  1. `labels` – список из `None` (точка еще не просмотрена); соседей всех точек посчитать заранее;
  2. для каждой непросмотренной точки `i`: если у нее меньше `min_samples` соседей – пометить `-1` и идти дальше; иначе завести новый кластер `cluster_id`, пометить `i` и положить ее соседей в очередь;
  3. обход очереди: точку `j` с меткой `-1` переводим в текущий кластер (это пограничная точка); уже размеченную пропускаем; новую помечаем текущим кластером, и **если она основная**, добавляем ее соседей в конец очереди.
- `point_types(X, labels, eps, min_samples)` – для каждой точки строка `'основная'`, `'пограничная'` или `'шум'`.

Очередь удобно обходить с указателем: `pos = 0; while pos < len(queue): j = queue[pos]; pos += 1; ...`.

In [ ]:
def region_query(X, i, eps):
    return ...                                  # <-- точки в eps-окрестности (включая i)

def dbscan(X, eps, min_samples):
    labels = [None] * len(X)
    neighbors = [region_query(X, i, eps) for i in range(len(X))]
    cluster_id = -1
    for i in range(len(X)):
        if labels[i] is not None:
            continue
        if len(neighbors[i]) < min_samples:
            labels[i] = ...                     # <-- пока считаем шумом
            continue
        cluster_id += 1
        labels[i] = cluster_id
        queue, pos = list(neighbors[i]), 0
        while pos < len(queue):
            j = queue[pos]
            pos += 1
            if labels[j] == -1:
                labels[j] = ...                 # <-- шум оказался пограничной точкой
            if labels[j] is not None:
                continue
            labels[j] = ...                     # <-- новая точка кластера
            if len(neighbors[j]) >= min_samples:
                ...                             # <-- основная: продолжаем обход
    return labels

def point_types(X, labels, eps, min_samples):
    types = []
    for i in range(len(X)):
        if len(region_query(X, i, eps)) >= min_samples:
            ...                                 # <-- плотная окрестность
        elif labels[i] != -1:
            ...                                 # <-- не основная, но в кластере
        else:
            types.append('шум')
    return types


P = [[0], [1], [2], [3], [6], [10], [11], [12]]
lab = dbscan(P, eps=1, min_samples=3)
print(lab, '  ожидается [0, 0, 0, 0, -1, 1, 1, 1]')
print(point_types(P, lab, 1, 3))
print("  ожидается: пограничная, основная, основная, пограничная, шум, пограничная, основная, пограничная")

**Вопрос Г5.** а) Почему точка 0 – пограничная, хотя она в кластере? Почему точка 6 – шум? б) Что изменится, если взять `min_samples = 4`? в) Точка 0 сначала могла быть помечена как шум, а потом стала пограничной. Почему пограничную точку нельзя «продолжать» – добавлять в очередь ее соседей?

*Ваш ответ:*

**Задание Г6. «Полумесяцы»: DBSCAN против k средних.**

Сравните на 200 точках «полумесяцев» метод k средних ($k = 2$) и DBSCAN (`eps = 0.25`, `min_samples = 5`) по ARI и по силуэту. Затем сверьте DBSCAN с реализацией из sklearn.

In [ ]:
c_km, l_km, J_km = kmeans_restarts(X_moons, 2)
l_db = ...                                      # <-- DBSCAN на полумесяцах
print(f'k средних: ARI {adjusted_rand_index(l_km, y_moons):.3f}, силуэт {silhouette_score(X_moons, l_km):.3f}')
print(f'DBSCAN:    ARI {adjusted_rand_index(l_db, y_moons):.3f}, силуэт {silhouette_score(X_moons, l_db):.3f}, '
      f'кластеров {len(set(l_db) - {-1})}, шумовых точек {l_db.count(-1)}')
print('  ожидается: k средних – 0.246 и 0.482;  DBSCAN – 1.000 и 0.338, 2 кластера, 0 шумовых точек')
for eps in (0.15, 0.2):
    l = dbscan(X_moons, eps, 5)
    print(f'eps = {eps}: кластеров {len(set(l) - {-1})}, шумовых точек {l.count(-1)}, ARI {adjusted_rand_index(l, y_moons):.3f}')
print('  ожидается: eps = 0.15 – 7 кластеров, 16 шумовых, 0.380;  eps = 0.2 – 3 кластера, 4 шумовых, 0.730')

In [ ]:
# Готовая ячейка: сверка со sklearn и рисунок
from sklearn.cluster import DBSCAN
sk = DBSCAN(eps=0.25, min_samples=5).fit(X_moons).labels_.tolist()
print(f'совпадение с sklearn (ARI наших меток и меток sklearn): {adjusted_rand_index(l_db, sk):.3f}   ожидается 1.000')
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
plot_clusters(ax[0], X_moons, l_km, 'k средних, k = 2')
plot_clusters(ax[1], X_moons, l_db, 'DBSCAN, eps = 0.25')
plot_clusters(ax[2], X_moons, dbscan(X_moons, 0.15, 5), 'DBSCAN, eps = 0.15: слишком мелко')
plt.tight_layout(); plt.show()

**Вопрос Г6.** а) DBSCAN разделил полумесяцы без ошибок, а его силуэт (0,338) **ниже**, чем у неверного разбиения k средних (0,482). Как это объяснить? Какой вывод о силуэте? б) Почему при `eps = 0.15` получилось 7 кластеров и много шума? в) Почему DBSCAN не нужно задавать число кластеров, а k средних нужно?

*Ваш ответ:*

**Задание Г7. Ложные отражения лидара: подбор `eps` и DBSCAN на сцене с шумом.**

Добавим на парковку 40 ложных отражений – случайных точек по всей площадке (`make_scene(n_noise=40)`, у них `y = -1`).

1. `k_distances(X, k)` – для каждой точки расстояние до ее $k$-го ближайшего соседа (сама точка – первый сосед), отсортированные по возрастанию (п. 5 теории). Посмотрите на график и выберите `eps` около «локтя».
2. Запустите DBSCAN с `min_samples = 8` и выбранным `eps = 1.0`. Посчитайте, сколько из 40 ложных отражений метод отметил шумом и сколько точек объектов отметил шумом по ошибке.
3. Сравните с методом k средних при $k = 6$: куда попадают ложные отражения у него?

In [ ]:
X_noisy, y_noisy = make_scene(n_noise=40)

def k_distances(X, k):
    result = []
    for i in range(len(X)):
        d = sorted(dist(X[i], X[j]) for j in range(len(X)))   # расстояния до всех точек, первое – 0 до самой себя
        ...                                     # <-- расстояние до k-го соседа
    return ...                                  # <-- по возрастанию


kd = k_distances(X_noisy, 8)
print('точек:', len(kd), '  k-расстояния в долях 50 %, 85 %, 95 %:', [round(kd[int(q * (len(kd) - 1))], 2) for q in (0.5, 0.85, 0.95)])

l_noisy = ...                                   # <-- DBSCAN на сцене с шумом
caught = ...                                    # <-- ложные отражения в шуме
lost = ...                                      # <-- точки объектов в шуме
print(f'DBSCAN: объектов {len(set(l_noisy) - {-1})}, шумом отмечено {l_noisy.count(-1)}: '
      f'{caught} из 40 ложных отражений и {lost} точки объектов; ARI {adjusted_rand_index(l_noisy, y_noisy):.3f}')
print('  ожидается: 6 объектов, 36 шумовых: 32 из 40 и 4 точки объектов; ARI 0.920')

c6, l6, J6 = kmeans_restarts(X_noisy, 6)
print(f'k средних, k = 6: ARI {adjusted_rand_index(l6, y_noisy):.3f}; '
      f'все 40 ложных отражений приписаны объектам   ожидается 0.809')

In [ ]:
# Готовая ячейка: график k-расстояний и результат
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].plot(range(len(kd)), kd, color=BLUE)
ax[0].axhline(1.0, color=NAVY, linestyle='--', label='eps = 1.0')
ax[0].set_xlabel('точки, по возрастанию'); ax[0].set_ylabel('расстояние до 8-го соседа, м')
ax[0].set_title('График k-расстояний'); ax[0].legend(); ax[0].grid(alpha=0.3)
plot_clusters(ax[1], X_noisy, l_noisy, 'DBSCAN: крестики – шум')
plot_clusters(ax[2], X_noisy, l6, 'k средних: шум внутри объектов')
plt.tight_layout(); plt.show()

**Вопрос Г7.** а) Почему 8 ложных отражений DBSCAN не нашел? Посмотрите, где они лежат. б) Почему 4 точки настоящих объектов попали в шум? Как это исправить и чем придется заплатить? в) Почему для беспилотника важно, что метод умеет говорить «эта точка ни к чему не относится»?

*Ваш ответ:*

---
### Задания повышенной сложности

1. **Аномалии в показаниях датчиков.** Готовый генератор `make_drive_log` в ячейке решения создает журнал поездки: 400 моментов времени, в каждом – скорость по спутниковой навигации, скорость по датчикам колес и продольное ускорение. В 12 моментов один из датчиков дает сбой (скачок скорости, проскальзывание колеса, удар по акселерометру). Стандартизуйте признаки, найдите аномалии как шумовые точки DBSCAN и посчитайте, сколько из 12 сбоев найдено и сколько нормальных моментов отмечено ложно. Как меняется результат при разных `eps`?
2. **Эффект цепочки.** Добавьте к 100 точкам «полумесяцев» 10 случайных точек в промежуток между ними и снова запустите одиночную связь. Что произошло с ARI? Справляется ли DBSCAN?
3. **Связь Уорда.** Расстояние между кластерами – прирост инерции при их объединении: $\frac{n_A n_B}{n_A + n_B}\|\mu_A - \mu_B\|^2$. Реализуйте его (храните центры и размеры кластеров) и сравните результат на парковке с методом k средних.
4. **Разная плотность.** Сделайте сцену, где один объект плотный (100 точек в радиусе 0,5 м), а другой разреженный (20 точек в радиусе 2 м). Найдется ли такой `eps`, при котором DBSCAN находит оба объекта и не находит лишних? Почитайте про метод HDBSCAN и объясните, как он решает эту проблему.
5. **Силуэт на рисунке.** Постройте для лучшего разбиения Wine ($k = 3$) «график силуэтов»: для каждого кластера отсортированные значения $s(i)$ в виде горизонтальных полос. Какие вина лежат на границах сортов?

In [ ]:
# Место для заданий повышенной сложности